In [1]:
!apt-get update -qq
!apt-get install -y flex bison gcc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gcc is already the newest version (4:11.2.0-1ubuntu1).
gcc set to manually installed.
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 78 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 

In [2]:
%%writefile art_expr.l
%{
#include <stdio.h>
#include "art_expr.tab.h"
%}

%%

[a-zA-Z][0-9a-zA-Z]* {
    return ID;
}

[0-9]+ {
    return DIG;
}

[ \t]+ {
    /* ignore spaces and tabs */
}

. {
    return yytext[0];
}

\n {
    return 0;
}

%%

int yywrap()
{
    return 1;
}

Writing art_expr.l


In [3]:
%%writefile art_expr.y
%{
#include <stdio.h>
#include <stdlib.h>

int yylex(void);
int yyerror(char *s);
%}

%token ID DIG

%left '+' '-'
%left '*' '/'
%right UMINUS

%%

stmt:
    expn
    ;

expn:
      expn '+' expn
    | expn '-' expn
    | expn '*' expn
    | expn '/' expn
    | '-' expn %prec UMINUS
    | '(' expn ')'
    | DIG
    | ID
    ;

%%

int main()
{
    printf("Enter the Expression:\n");

    if (yyparse() == 0)
        printf("Valid Expression\n");

    return 0;
}

int yyerror(char *s)
{
    printf("Invalid Expression\n");
    return 0;
}

Writing art_expr.y


In [4]:
!rm -f art_expr.tab.c art_expr.tab.h lex.yy.c art_expr

In [5]:
!bison -d art_expr.y

In [6]:
!flex art_expr.l

In [7]:
!gcc lex.yy.c art_expr.tab.c -o art_expr -lfl

In [8]:
!ls -l art_expr

-rwxr-xr-x 1 root root 31888 Aug 25 14:13 art_expr


In [9]:
!echo "a+b" | ./art_expr

Enter the Expression:
Valid Expression


In [10]:
!echo "a*b+c" | ./art_expr

Enter the Expression:
Valid Expression
